# Deep Results Analysis (Modeling Outputs)

This notebook is focused on **`outputs/modeling`** and provides:
- clean leaderboard tables with uncertainty
- experiment stability diagnostics across seeds
- model-family and variant comparisons
- statistically grounded interpretation for slang-heavy finetuning

The goal is to produce publication-ready figures, tables, and clear explanations directly from project outputs.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

cwd = Path.cwd()
if (cwd / 'outputs' / 'modeling').exists():
    ROOT = cwd
else:
    candidates = [cwd, *cwd.parents]
    ROOT = next((p for p in candidates if (p / 'outputs' / 'modeling').exists()), cwd)

MODEL_OUT = ROOT / 'outputs' / 'modeling'
assert MODEL_OUT.exists(), f'Expected modeling outputs at: {MODEL_OUT}'

RUN_PATH = MODEL_OUT / 'run_summary_table.csv'
MULTI_PATH = MODEL_OUT / 'multiseed_summary_table.csv'
PAIR_A_PATH = MODEL_OUT / 'paired_significance_bert_vs_finetune_slang_seed42.json'
PAIR_B_PATH = MODEL_OUT / 'paired_significance_bertmixed_vs_finetune_mixed_seed42.json'

for p in [RUN_PATH, MULTI_PATH, PAIR_A_PATH, PAIR_B_PATH]:
    assert p.exists(), f'Missing expected file: {p}'

MODEL_OUT


In [ ]:
# Plot dependencies (install on demand inside notebook kernel)
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    %pip install -q matplotlib
    import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

print('Notebook plotting environment ready.')


In [ ]:
pair_a = json.loads(PAIR_A_PATH.read_text(encoding='utf-8'))
pair_b = json.loads(PAIR_B_PATH.read_text(encoding='utf-8'))

comparison_meta = pd.DataFrame(
    [
        {
            'comparison': 'bert_base_baseline vs bert_finetune_slang_heavy',
            'matched_examples': pair_a['matched_examples'],
            'acc_delta_finetune_minus_baseline': pair_a['accuracy']['difference_b_minus_a'],
            'mcnemar_p': pair_a['tests']['mcnemar_chi_square_pvalue'],
            'sign_test_p': pair_a['tests']['sign_test_two_sided_pvalue'],
            'bootstrap_ci_low': pair_a['bootstrap_accuracy_diff_b_minus_a']['ci95_low'],
            'bootstrap_ci_high': pair_a['bootstrap_accuracy_diff_b_minus_a']['ci95_high'],
        },
        {
            'comparison': 'bert_base_mixed vs bert_finetune_slang_heavy_mixed',
            'matched_examples': pair_b['matched_examples'],
            'acc_delta_finetune_minus_baseline': pair_b['accuracy']['difference_b_minus_a'],
            'mcnemar_p': pair_b['tests']['mcnemar_chi_square_pvalue'],
            'sign_test_p': pair_b['tests']['sign_test_two_sided_pvalue'],
            'bootstrap_ci_low': pair_b['bootstrap_accuracy_diff_b_minus_a']['ci95_low'],
            'bootstrap_ci_high': pair_b['bootstrap_accuracy_diff_b_minus_a']['ci95_high'],
        },
    ]
)

comparison_meta


In [ ]:
run_df_raw = pd.read_csv(RUN_PATH)
multi_df = pd.read_csv(MULTI_PATH)

# Remove repeated seed rows by keeping the best available row per experiment+seed.
# This protects analysis from accidental duplicate log exports.
run_df = (
    run_df_raw
    .sort_values(['experiment_name', 'seed', 'test_macro_f1'], ascending=[True, True, False])
    .drop_duplicates(subset=['experiment_name', 'seed'], keep='first')
    .copy()
)

def infer_family(name: str) -> str:
    if 'bertweet' in name:
        return 'bertweet'
    if 'gpt' in name:
        return 'gpt'
    return 'bert'

def infer_variant(name: str) -> str:
    if 'slang_masked' in name:
        return 'slang_masked'
    if 'mixed' in name:
        return 'mixed'
    return 'original'

for df in (run_df, multi_df):
    df['model_family'] = df['experiment_name'].map(infer_family)
    df['variant'] = df['experiment_name'].map(infer_variant)

run_df['seed'] = run_df['seed'].astype(str)
run_df['family_variant'] = run_df['model_family'] + ' | ' + run_df['variant']

print(f'run_summary rows: raw={len(run_df_raw)}, deduplicated={len(run_df)}')
run_df[['experiment_name', 'seed', 'model_name', 'test_accuracy', 'test_macro_f1']].head(12)


In [ ]:
leaderboard = (
    multi_df[['experiment_name', 'model_family', 'variant', 'test_accuracy_mean', 'test_accuracy_std', 'test_macro_f1_mean', 'test_macro_f1_std']]
    .sort_values('test_macro_f1_mean', ascending=False)
    .reset_index(drop=True)
)
leaderboard.index = leaderboard.index + 1
leaderboard.index.name = 'rank'

leaderboard_display = leaderboard.copy()
for col in ['test_accuracy_mean', 'test_accuracy_std', 'test_macro_f1_mean', 'test_macro_f1_std']:
    leaderboard_display[col] = leaderboard_display[col].map(lambda x: f'{x:.4f}')

leaderboard_display


In [ ]:
# Leaderboard plot with uncertainty bars and family color coding
plot_df = leaderboard.copy()
family_colors = {'bert': '#1f77b4', 'bertweet': '#2ca02c', 'gpt': '#ff7f0e'}
colors = [family_colors.get(f, '#7f7f7f') for f in plot_df['model_family']]

plt.figure(figsize=(13, 6))
bars = plt.bar(
    x=plot_df['experiment_name'],
    height=plot_df['test_macro_f1_mean'],
    yerr=plot_df['test_macro_f1_std'],
    capsize=4,
    color=colors,
    alpha=0.90,
    edgecolor='black',
    linewidth=0.6,
    error_kw={'elinewidth': 1.0, 'alpha': 0.9},
)

for bar, val in zip(bars, plot_df['test_macro_f1_mean']):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        val + 0.005,
        f'{val:.3f}',
        ha='center',
        va='bottom',
        fontsize=9,
    )

plt.xticks(rotation=35, ha='right')
plt.ylabel('Macro-F1 mean')
plt.title('Multiseed Macro-F1 by Experiment (with Std Error Bars)')
legend_handles = [
    plt.Rectangle((0, 0), 1, 1, color=family_colors[k], label=k)
    for k in ['bert', 'bertweet', 'gpt']
    if k in plot_df['model_family'].values
]
plt.legend(handles=legend_handles, title='Model family', frameon=True)
plt.ylim(0.35, min(1.0, plot_df['test_macro_f1_mean'].max() + 0.08))
plt.tight_layout()
plt.show()


In [ ]:
# Compare core model families on comparable (non-finetune) experiments
core_multi = multi_df[~multi_df['experiment_name'].str.contains('finetune')].copy()

family_variant_table = (
    core_multi.pivot_table(
        index='model_family',
        columns='variant',
        values='test_macro_f1_mean',
        aggfunc='mean',
    )
    .reindex(index=['bert', 'bertweet'])
)

common_cols = [c for c in ['original', 'mixed', 'slang_masked'] if c in family_variant_table.columns]
bert_minus_bertweet = pd.DataFrame({
    'variant': common_cols,
    'bert_minus_bertweet_macro_f1': [
        family_variant_table.loc['bert', c] - family_variant_table.loc['bertweet', c]
        for c in common_cols
    ],
})

print('Macro-F1 by model family and variant (core experiments only):')
family_variant_table.round(4)


## Stability and Risk Diagnostics

The next cells evaluate **how reliable each experiment is across seeds**, not only how high the average score is.

Focus points:
- variance and coefficient of variation (CV) across seeds
- per-seed score spread by experiment
- quick detection of fragile settings (high volatility)

This is important because a high mean can still hide a reproducibility risk if seed sensitivity is large.


In [ ]:
stability = (
    run_df.groupby('experiment_name', as_index=False).agg(
        macro_f1_mean=('test_macro_f1', 'mean'),
        macro_f1_std=('test_macro_f1', 'std'),
        accuracy_mean=('test_accuracy', 'mean'),
        accuracy_std=('test_accuracy', 'std'),
        seed_count=('seed', 'nunique'),
    )
    .sort_values('macro_f1_std', ascending=False)
    .reset_index(drop=True)
)

stability['macro_f1_cv_pct'] = (100 * stability['macro_f1_std'] / stability['macro_f1_mean']).round(2)
stability['accuracy_cv_pct'] = (100 * stability['accuracy_std'] / stability['accuracy_mean']).round(2)

stability_display = stability.copy()
for col in ['macro_f1_mean', 'macro_f1_std', 'accuracy_mean', 'accuracy_std']:
    stability_display[col] = stability_display[col].map(lambda x: f'{x:.4f}')

print('Stability table (sorted by Macro-F1 std descending):')
stability_display


In [ ]:
# Seed-level heatmap and spread profile
heat = run_df.pivot_table(
    index='experiment_name',
    columns='seed',
    values='test_macro_f1',
    aggfunc='mean',
)
first_seed = sorted(heat.columns)[0]
heat = heat.sort_values(by=first_seed, ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

im = axes[0].imshow(heat.values, aspect='auto', cmap='viridis')
axes[0].set_xticks(range(len(heat.columns)))
axes[0].set_xticklabels([str(c) for c in heat.columns])
axes[0].set_yticks(range(len(heat.index)))
axes[0].set_yticklabels(list(heat.index))
axes[0].set_xlabel('Seed')
axes[0].set_ylabel('Experiment')
axes[0].set_title('Seed-Level Macro-F1 Heatmap')
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        val = heat.iloc[i, j]
        label = 'NA' if pd.isna(val) else f'{val:.3f}'
        axes[0].text(j, i, label, ha='center', va='center', color='white', fontsize=8)
fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04, label='Macro-F1')

box_order = leaderboard['experiment_name'].tolist()
data = [run_df.loc[run_df['experiment_name'] == exp, 'test_macro_f1'].values for exp in box_order]
axes[1].boxplot(data, tick_labels=box_order, showmeans=True)
axes[1].set_xticklabels(box_order, rotation=35, ha='right')
axes[1].set_ylabel('Macro-F1 across seeds')
axes[1].set_title('Seed Spread by Experiment')

plt.tight_layout()
plt.show()


In [ ]:
# Statistical significance summary for slang-heavy finetuning comparisons
sig_table = comparison_meta.copy()
sig_table['significant_0.05'] = (sig_table['mcnemar_p'] < 0.05) & (sig_table['sign_test_p'] < 0.05)
sig_table['effect_direction'] = np.where(
    sig_table['acc_delta_finetune_minus_baseline'] < 0,
    'finetune worse',
    'finetune better or equal',
)

sig_table_display = sig_table.copy()
for col in ['acc_delta_finetune_minus_baseline', 'bootstrap_ci_low', 'bootstrap_ci_high']:
    sig_table_display[col] = sig_table_display[col].map(lambda x: f'{x:.4f}')
for col in ['mcnemar_p', 'sign_test_p']:
    sig_table_display[col] = sig_table_display[col].map(lambda x: f'{x:.2e}')

sig_table_display


In [ ]:
# Effect-size visualization with bootstrap confidence intervals
viz = comparison_meta.copy()
viz['ci_center'] = (viz['bootstrap_ci_low'] + viz['bootstrap_ci_high']) / 2
viz['ci_halfwidth'] = (viz['bootstrap_ci_high'] - viz['bootstrap_ci_low']) / 2

plt.figure(figsize=(11, 5))
plt.axhline(0.0, color='black', linestyle='--', linewidth=1)
plt.errorbar(
    x=viz['comparison'],
    y=viz['acc_delta_finetune_minus_baseline'],
    yerr=viz['ci_halfwidth'],
    fmt='o',
    capsize=5,
    color='#d62728',
    ecolor='#555555',
)

for i, row in viz.iterrows():
    plt.text(
        i,
        row['acc_delta_finetune_minus_baseline'] - 0.003,
        f"delta={row['acc_delta_finetune_minus_baseline']:.3f}\nCI[{row['bootstrap_ci_low']:.3f}, {row['bootstrap_ci_high']:.3f}]",
        ha='center',
        va='top',
        fontsize=9,
    )

plt.xticks(rotation=20, ha='right')
plt.ylabel('Accuracy delta (finetune - baseline)')
plt.title('Slang-Heavy Finetuning Impact with 95% Bootstrap CI')
plt.tight_layout()
plt.show()


In [ ]:
# Executive summary generated from computed tables
top = leaderboard.iloc[0]
most_unstable = stability.iloc[0]
largest_gap = bert_minus_bertweet.sort_values('bert_minus_bertweet_macro_f1', ascending=False).iloc[0]

print('Key findings:')
print(f"1) Best experiment: {top['experiment_name']} (Macro-F1={top['test_macro_f1_mean']:.4f} +/- {top['test_macro_f1_std']:.4f}).")
print(f"2) Highest seed instability: {most_unstable['experiment_name']} (Macro-F1 std={most_unstable['macro_f1_std']:.4f}, CV={most_unstable['macro_f1_cv_pct']:.2f}%).")
print(f"3) Largest BERT-BERTweet gap: {largest_gap['variant']} variant (delta={largest_gap['bert_minus_bertweet_macro_f1']:.4f}).")

all_negative = (comparison_meta['acc_delta_finetune_minus_baseline'] < 0).all()
all_sig = ((comparison_meta['mcnemar_p'] < 0.05) & (comparison_meta['sign_test_p'] < 0.05)).all()
print(f"4) Slang-heavy finetuning deltas are all negative: {all_negative}; statistically significant in both tests: {all_sig}.")
